# CNN Training on ImageNet-100
- Single recipe, from scratch, for Pure Head-to-Head Architectural comparison.
- Models: timm first, torchvision as fallback. 
- Only models without a completed, recipe-matching entry in `cnn_training.csv` are trained. Interrupted runs resume from `<model>.resume.pt`.
- Outputs: `checkpoint/pytorch/<model>.pt`, `data/cnn-train-data/cnn_training.csv` (one row per model), `cnn_training_epochs.csv` (one row per epoch).
- Data path: raw ImageFolder -> one-time uint8 memmap cache -> GPU-side augmentation (avoids the 6-core JPEG decoding bottleneck).

In [1]:
# !pip install -q timm tqdm pandas   # uncomment on first run
import os, io, json, time, math, copy, gc, queue, hashlib, platform, threading, traceback
from dataclasses import dataclass, field, asdict, replace
from pathlib import Path
from typing import Optional, Any

import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, timm
import torchvision.transforms as T
from torchvision.ops import roi_align
from timm.data import Mixup
from tqdm.auto import tqdm

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch: {torch.__version__}\ntimm: {timm.__version__}\ntorchvision: {torchvision.__version__}\nGPU-Device: {DEVICE}",
      torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

torch: 2.11.0+cu128
timm: 1.0.30
torchvision: 0.26.0+cu128
GPU-Device: cuda NVIDIA GeForce RTX 4060 Ti


C:\Users\chabz\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Paths and configuration

In [6]:
ROOT        = Path.cwd().resolve().parent          # notebook lives in ai-accelerators/notebooks
DATASET_DIR = ROOT / "datasets" / "imageNet-100"   # expects train/<class>/*.JPEG and val/<class>/*.JPEG
SPLIT_DIRS = {
    "train": ["train.X1", "train.X2", "train.X3", "train.X4"],
    "val":   ["val.X"],
}
CACHE_DIR   = ROOT / "datasets" / "imageNet-100_cache"
CKPT_DIR    = ROOT / "checkpoint" / "pytorch"
DATA_DIR    = ROOT / "data" / "cnn-train-data"
NUM_WORKERS = min(6, os.cpu_count() or 1)          # only used for building the cache

# Defining Traing Configuration that applies to all models. This Allows Pure Architectural Comparison of the Models

@dataclass
class TrainConfig:
    num_classes: int = 100           #InageNet-100 has 100 classes. Change for a different Dataset.
    epochs: int = 100                # Same Epochs fro all Models.
    batch_size: int = 256            # effective batch size (micro-batch x accumulation)
    lr: float = 2e-3                 # AdamW peak LR for batch_size above
    weight_decay: float = 0.05       # not applied to BN / bias
    warmup_epochs: int = 5
    min_lr: float = 1e-5
    label_smoothing: float = 0.1
    mixup_alpha: float = 0.1         # set both alphas to 0 to disable mixup/cutmix
    cutmix_alpha: float = 1.0
    mix_prob: float = 0.5
    train_res: int = 160             # train resolution == cache resolution
    eval_res: int = 224              # val resolution (resize eval_res/0.875, center crop eval_res)
    rrc_scale: tuple = (0.35, 1.0)   # RandomResizedCrop scale range (RSB-A3 style)
    clip_grad: float = 5.0
    seed: int = 42                   # Same RandomSeed for result reproducibility
    amp_dtype: str = "bfloat16"
    compile: bool = False            # torch.compile (slow warmup per model; excluded from recipe hash)
    limit_batches: Optional[int] = None   # smoke tests only
    debug: bool = False              # debug runs write to separate _debug folders / files

    def recipe_hash(self) -> str:
        d = asdict(self)
        for k in ("compile", "limit_batches", "debug"): # remove uneeded configs
            d.pop(k)
        return hashlib.md5(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:8] #specific HASH code for the training Recipe.



In [7]:
CFG = TrainConfig()
print(CFG)

CFG_HASH = CFG.recipe_hash()
print(CFG_HASH)

TrainConfig(num_classes=100, epochs=100, batch_size=256, lr=0.002, weight_decay=0.05, warmup_epochs=5, min_lr=1e-05, label_smoothing=0.1, mixup_alpha=0.1, cutmix_alpha=1.0, mix_prob=0.5, train_res=160, eval_res=224, rrc_scale=(0.35, 1.0), clip_grad=5.0, seed=42, amp_dtype='bfloat16', compile=False, limit_batches=None, debug=False)
397cfed2


## 2. Dataset cache (one-time) and GPU data loader

In [9]:
#This Class is a map-style PyTorch Dataset. There is Nothing that is loaded from the Disk when its created, 
# except the directory listsing. 
# It holds a list of (file_paths, integer_label) pairs instead of load all 130k images into the Disk.
# The image is only loaded into the disk, by DataLoader class when the index points to this stored dataset.
# 

class MultiRootImageFolder(torch.utils.data.Dataset):
    """ImageFolder over several root dirs; class folders with the same name are merged."""
    EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    def __init__(self, roots, classes=None, transform=None):
        roots = [Path(r) for r in roots]
        for r in roots: assert r.is_dir(), f"missing folder: {r}"
        found = sorted({d.name for r in roots for d in r.iterdir() if d.is_dir()})
        self.classes = classes if classes is not None else found
        assert set(found) == set(self.classes), (
            f"class mismatch: {len(found)} found vs {len(self.classes)} expected")
        cidx = {c: i for i, c in enumerate(self.classes)}
        self.samples = sorted(
            (str(p), cidx[p.parent.name])
            for r in roots for p in r.rglob("*")
            if p.suffix.lower() in self.EXT and p.parent.name in cidx)
        self.transform = transform
        # This is deafult torchvision's loader. It opens the file with PIL and calls .covert("RGB").
        # This is important for ImageNet, which contains a few grayscale and CMYK image files, which would
        # lead to variable image channels, possibly crashing batching.
        self.loader = torchvision.datasets.folder.default_loader   # PIL -> RGB
    def __len__(self): return len(self.samples)

    # This looks up (path,label) and decodes JPEG into a PIL image (this is expensive step costing few ms of CPU)
    # It then applies the given transform(transform reshapes ot 160x160).
    # The result is a uint8 tensor of shape (3, 160,160), plus an int label.
    # why resize from 224px to 160px during training? 
    # Paper: Touvron et al. (2019), “Fixing the Train-Test Resolution Discrepancy explains why.
    # Short Answer: Random-resized-crop zooms objects in during training, 
    #so objects look larger in training than at test time. 
    #Testing at a somewhat higher resolution than the training one compensates for this
    def __getitem__(self, i):
        path, y = self.samples[i]
        img = self.loader(path)
        return (self.transform(img) if self.transform else img), y

# This function converts training data (about 130K images) JPEGs into two file":
    # train_160_x.npy: one uint8 array of shape (N,160,160,3) every image already resized and cropped.
    # train_160_y.npy: integer labels
# It only runs once, to later runs skip it.
# This Catch is necessary because decoding JPEG and resizing it during training costs few milliseconds of the GPU,
# This cache pays the decode cost once, after which an epoch is just reading raw bytes from the file.
# the CACHE directory is not tracked by git.

# Information Loss: Training images are centre-cropped to 160 x 160 once, so each train images has a fixed crop
# the edges of non-square images are gone. Data Argumentation later crops within that 160 x 160 square. this is a
# delibarate trade-of of augmentation variety ofr speed.
def build_cache(split: str, res: int, resize_to: int):
    """Multi-folder ImageNet-100 -> uint8 NHWC .npy memmap + labels. Skipped if it already exists."""
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    fx, fy = CACHE_DIR / f"{split}_{res}_x.npy", CACHE_DIR / f"{split}_{res}_y.npy"
    if fx.exists() and fy.exists():
        return fx, fy
    cls_file = CACHE_DIR / "classes.json"
    classes = json.loads(cls_file.read_text()) if cls_file.exists() else None
    tf = T.Compose([T.Resize(resize_to), T.CenterCrop(res), T.PILToTensor()])
    ds = MultiRootImageFolder([DATASET_DIR / d for d in SPLIT_DIRS[split]], classes, tf)
    assert len(ds.classes) == CFG.num_classes, f"{split}: found {len(ds.classes)} classes"
    print(f"{split}: {len(ds)} images, {len(ds.classes)} classes")
    if classes is None:
        cls_file.write_text(json.dumps(ds.classes))   # train runs first, so train defines the class order
    dl = torch.utils.data.DataLoader(ds, batch_size=256, num_workers=NUM_WORKERS, shuffle=False)
    tmp = CACHE_DIR / f"{split}_{res}_x.tmp.npy"
    mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.uint8, shape=(len(ds), res, res, 3))
    ys, i = [], 0
    for x, y in tqdm(dl, desc=f"cache {split}@{res}"):
        mm[i:i + len(x)] = x.permute(0, 2, 3, 1).numpy(); i += len(x); ys.append(y)
    mm.flush(); del mm
    np.save(fy, torch.cat(ys).numpy().astype(np.int64))
    os.replace(tmp, fx)
    return fx, fy

# This function runs the build_cache function for training and validation. val images are scaled to  256 from 224.
def ensure_caches(cfg: TrainConfig):
    build_cache("train", cfg.train_res, cfg.train_res)
    build_cache("val", cfg.eval_res, int(round(cfg.eval_res / 0.875)))

#This are standard ImageNet-1K statistics. They are per-channel, since ImageNet-100 is randomish subset of this
# it shoulf have roughly similar statistics.
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1) * 255
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1) * 255

def normalize(x):  # x float NCHW in [0,255]
    return (x - MEAN.to(x.device)) / STD.to(x.device)

# This function applies two standard ImageNet training augmentations to the whole batch on GPU.
# there are RandomResizedCrop ( givining each image a random crop covering 35-100% of its area)
# and horizontal flip with probablity of 0.5.

#The cache stores fixed, pre-cropped uint8 images, 
#so the usual torchvision augmentation pipeline (which runs on the CPU per image) would bring back 
#the CPU bottleneck we built the cache to avoid. Doing it on the GPU makes augmentation nearly free.
def gpu_train_aug(x_u8, out_res, scale, ratio=(3/4, 4/3)):
    """Per-sample RandomResizedCrop (via roi_align) + horizontal flip, entirely on GPU."""
    n, s = x_u8.shape[0], x_u8.shape[1]
    x = x_u8.permute(0, 3, 1, 2).float(); dev = x.device
    area = s * s * torch.empty(n, device=dev).uniform_(*scale)
    ar = torch.exp(torch.empty(n, device=dev).uniform_(math.log(ratio[0]), math.log(ratio[1])))
    w = torch.sqrt(area * ar).clamp(max=s); h = torch.sqrt(area / ar).clamp(max=s)
    x0 = torch.rand(n, device=dev) * (s - w); y0 = torch.rand(n, device=dev) * (s - h)
    boxes = torch.stack([torch.arange(n, device=dev, dtype=x.dtype), x0, y0, x0 + w, y0 + h], 1)
    x = roi_align(x, boxes, (out_res, out_res), spatial_scale=1.0, sampling_ratio=2, aligned=True)
    flip = torch.rand(n, device=dev) < 0.5
    return torch.where(flip[:, None, None, None], x.flip(3), x)


# This function serves mini-batches from the cached .npy files, replacing PyTorch's DataLoader. 
# Each iteration yields a batch of raw uint8 images (NHWC) and int64 labels, both already pinned to memory
# ready to copy to the GPU

# How is Works:
# Memeory-mapped array does not laod the 10GB train cache into RAM. The OS pages in only what is touched,
# and after the first epoch most of it sits in the page cache, so later epochs read from RAM, rather them from
# Disk.

# Each epoch gets a new random permutation of the sample indeces, seeded with seed + epoch. 
# The order is random across epoches, but still reproducible.

# This is the second half of the speed solution. The cache defined above removes JPEG decoding bottleneck, and 
# this class removes per-class Python overhead of a normal Dataset + DataLoader. A batch  simply one fancy-indexing
# call that copies about 20MB, making the laoder to cost very little on a 6-core CPU while the GPU stays busy.

class CachedLoader:
    """Memmap-backed loader with a background prefetch thread. Yields pinned uint8 NHWC + int64 labels."""
    def __init__(self, split, res, batch_size, train, seed=0, limit_batches=None, prefetch=4):
        self.x = np.load(CACHE_DIR / f"{split}_{res}_x.npy", mmap_mode="r")
        self.y = torch.from_numpy(np.load(CACHE_DIR / f"{split}_{res}_y.npy"))
        self.bs, self.train, self.seed, self.prefetch = batch_size, train, seed, prefetch
        self.n = len(self.y); self.epoch = 0
        nb = self.n // batch_size if train else math.ceil(self.n / batch_size)
        self.nb = min(nb, limit_batches) if limit_batches else nb
    def __len__(self): return self.nb
    def set_epoch(self, e): self.epoch = e
    def _produce(self, q):
        try:
            idx = (np.random.default_rng(self.seed + self.epoch).permutation(self.n)
                   if self.train else np.arange(self.n))
            for b in range(self.nb):
                bi = np.sort(idx[b * self.bs:(b + 1) * self.bs])   # sorted -> faster memmap reads
                q.put((torch.from_numpy(self.x[bi]).pin_memory(), self.y[bi].pin_memory()))
            q.put(None)
        except Exception as e:
            q.put(e)
    def __iter__(self):
        q = queue.Queue(self.prefetch)
        threading.Thread(target=self._produce, args=(q,), daemon=True).start()
        while True:
            item = q.get()
            if item is None: return
            if isinstance(item, Exception): raise item
            yield item

## 3. Model registry (timm first, torchvision fallback)

In [ ]:
# ADD / REMOVE MODELS HERE (string, or dict with overrides) 
# This designed to be as flexible as possble to allow for experimentation with different models
# Clever design.

In [13]:
MODELS = [
    "mobilenetv1_100",
    "mobilenetv2_100",
    "mobilenetv3_small_100",
    "mobilenetv3_large_100",
    "mobilenetv4_conv_small",
    "mobilenetv4_hybrid_medium", #this contains attention headers, which are interesting to investigate on TPU and 
    # Hailo to compile. Keep it for deployable test case.
    
    "efficientnet_lite0",
    "efficientnet_b0",
    
    "ghostnet_100",
    "ghostnetv2_100",
    "ghostnetv3_100",
    
    {"name": "shufflenet_v2_x1_0", "source": "torchvision"},
    #shufflenetv1 is  neither in torchvision.models nor timmn. will need to be sourced from external implementatoions
    
    "squeezenet1_0",
    "squeezenet1_1",#squeezeNext not found in torchvision.models nor timm, will need to be sourced from external repo.
]

In [14]:
#ModelSpec is a small record describing the model to train
# name: is the architecture
# source: forces or torchvision, and None means auto.
# kwargs: extra constructor arguments passed to model creating routine
# micro_batch_size: this is optional per-model override for the moels that run out of GPU memory.

@dataclass
class ModelSpec:
    name: str
    source: Optional[str] = None            # None = auto (timm -> torchvision); or "timm" / "torchvision"
    kwargs: dict = field(default_factory=dict)   # e.g. {"drop_path_rate": 0.1}
    micro_batch_size: Optional[int] = None  # set smaller than CFG.batch_size if OOM; accumulation is automatic

# This function contains plain strigs, dicts, or ModelSpec object and normalizes them all to ModelSpec
# This ensures that addind or removing a model is just one-line edit.
def as_spec(m) -> ModelSpec:
    return m if isinstance(m, ModelSpec) else ModelSpec(m) if isinstance(m, str) else ModelSpec(**m)

def build_model(spec: ModelSpec, num_classes: int):
    if spec.source in (None, "timm") and timm.is_model(spec.name):
        return timm.create_model(spec.name, pretrained=False, num_classes=num_classes, **spec.kwargs), "timm"
    if spec.source in (None, "torchvision") and spec.name in torchvision.models.list_models(module=torchvision.models):
        return torchvision.models.get_model(spec.name, weights=None, num_classes=num_classes, **spec.kwargs), "torchvision"
    raise ValueError(f"'{spec.name}' not found (source={spec.source})")


# This function returns model's size: parameters in millions, and MACs (which could translate to FLOPS) in billions.
# this is for one image at the given resolution.

# Parameters are counted direclty from the weight tensors.
# MACs are computeed from PyTorchs' FlopCounterMode. These are Theoretical Hardware Independent 
def complexity(model, res):
    from torch.utils.flop_counter import FlopCounterMode
    model = copy.deepcopy(model).eval().cpu()
    params = sum(p.numel() for p in model.parameters())
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        model(torch.zeros(1, 3, res, res))
    return params / 1e6, fc.get_total_flops() / 2 / 1e9   # (M params, GMACs)

# This function is just a try run on the model that does non training. For each model, it
# builds the model, runs complexity at a 224 image, and returns a table of source library, parameters, and GMACs.
def preview_models(models=MODELS, cfg=CFG):
    rows = []
    for m in map(as_spec, models):
        try:
            net, src = build_model(m, cfg.num_classes)
            p, g = complexity(net, 224)
            rows.append(dict(model=m.name, source=src, params_M=round(p, 2), GMACs_224=round(g, 3)))
        except Exception as e:
            rows.append(dict(model=m.name, source="NOT AVAILABLE", error=str(e)[:80]))
    return pd.DataFrame(rows)

## Comment of Theoretical MACSs: The FlopCounterMode counts convolutions, matrux multiplications, and attention
# but not BatchNorm, activations, pooling, concatenation, or channel shuffle. It matches "MAC" convention
# in the Literature, and a reason why MACs correlate poorly with latency on shuffle, depthwise-hevay models.

# Params include 100 classification head, so they are expected to be smaller than the 1000 published parameters.

## 4. Results store (CSV) — decides what still needs training

In [15]:

#This class is like a bookkeeping layer. It own the two CSVs files and the checkpoint locations
# It is implemented here to keep rack of the model's trained checkpoint, so that  models are not retrained on\
# every run. 

class ResultsStore:
    def __init__(self, cfg: TrainConfig):
        sub = "_debug" if cfg.debug else ""
        self.data_dir = DATA_DIR / sub if sub else DATA_DIR
        self.ckpt_dir = CKPT_DIR / sub if sub else CKPT_DIR
        self.data_dir.mkdir(parents=True, exist_ok=True); self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        self.summary_csv = self.data_dir / "cnn_training.csv"
        self.epochs_csv = self.data_dir / "cnn_training_epochs.csv"
    def _read(self, p): return pd.read_csv(p) if p.exists() else pd.DataFrame()
    def _write(self, df, p):
        tmp = p.with_suffix(".tmp"); df.to_csv(tmp, index=False); os.replace(tmp, p)
    def summary(self): return self._read(self.summary_csv)
    def epochs(self): return self._read(self.epochs_csv)
    def final_ckpt(self, name): return self.ckpt_dir / f"{name}.pt"
    def resume_ckpt(self, name): return self.ckpt_dir / f"{name}.resume.pt"
    def is_done(self, name, recipe_hash) -> bool:
        df = self.summary()
        if df.empty or not self.final_ckpt(name).exists(): return False
        r = df[(df.model == name) & (df.status == "completed")]
        return (not r.empty) and (r.iloc[-1].recipe_hash == recipe_hash)
    def upsert_summary(self, row: dict):
        df = self.summary()
        if not df.empty: df = df[df.model != row["model"]]
        self._write(pd.concat([df, pd.DataFrame([row])], ignore_index=True), self.summary_csv)
    def append_epoch(self, row: dict):
        self._write(pd.concat([self.epochs(), pd.DataFrame([row])], ignore_index=True), self.epochs_csv)
    def drop_epochs(self, name, after_epoch=0):
        df = self.epochs()
        if not df.empty:
            self._write(df[~((df.model == name) & (df.epoch > after_epoch))], self.epochs_csv)

## 5. ModelTrainer

In [16]:
class ModelTrainer:
    def __init__(self, spec: ModelSpec, cfg: TrainConfig, store: ResultsStore, device=DEVICE):
        self.spec, self.cfg, self.store, self.device = spec, cfg, store, device
        self.amp_dtype = getattr(torch, cfg.amp_dtype)
        self.micro_bs = spec.micro_batch_size or cfg.batch_size
        assert cfg.batch_size % self.micro_bs == 0, "batch_size must be divisible by micro_batch_size"
        self.accum = cfg.batch_size // self.micro_bs
        self.mixup = (Mixup(mixup_alpha=cfg.mixup_alpha, cutmix_alpha=cfg.cutmix_alpha, prob=cfg.mix_prob,
                            switch_prob=0.5, mode="batch", label_smoothing=cfg.label_smoothing,
                            num_classes=cfg.num_classes)
                      if (cfg.mixup_alpha > 0 or cfg.cutmix_alpha > 0) else None)

    # ---- helpers ----
    @staticmethod
    def _seed(s):
        import random; random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

    def _param_groups(self, model):
        decay, no_decay = [], []
        for n, p in model.named_parameters():
            if p.requires_grad: (no_decay if p.ndim <= 1 or n.endswith(".bias") else decay).append(p)
        return [{"params": decay, "weight_decay": self.cfg.weight_decay},
                {"params": no_decay, "weight_decay": 0.0}]

    def _lr_at(self, step, steps_per_epoch):
        c = self.cfg; warm = c.warmup_epochs * steps_per_epoch; total = c.epochs * steps_per_epoch
        if step < warm: return c.lr * (step + 1) / warm
        t = (step - warm) / max(1, total - warm)
        return c.min_lr + 0.5 * (c.lr - c.min_lr) * (1 + math.cos(math.pi * t))

    # ---- train / eval ----
    def _train_epoch(self, fmodel, opt, loader, epoch, step, steps_per_epoch):
        c = self.cfg; fmodel.train(); loader.set_epoch(epoch)
        loss_sum = torch.zeros((), device=self.device); n_micro = 0; seen = 0
        opt.zero_grad(set_to_none=True)
        for i, (x, y) in enumerate(loader):
            x = x.to(self.device, non_blocking=True); y = y.to(self.device, non_blocking=True)
            x = normalize(gpu_train_aug(x, c.train_res, c.rrc_scale)).contiguous(memory_format=torch.channels_last)
            if self.mixup is not None: x, t = self.mixup(x, y)
            with torch.autocast(self.device, dtype=self.amp_dtype):
                out = fmodel(x)
            out = out.float()
            loss = (F.cross_entropy(out, t) if self.mixup is not None
                    else F.cross_entropy(out, y, label_smoothing=c.label_smoothing))
            (loss / self.accum).backward()
            loss_sum += loss.detach(); n_micro += 1; seen += len(y)
            if (i + 1) % self.accum == 0:
                lr = self._lr_at(step, steps_per_epoch)
                for g in opt.param_groups: g["lr"] = lr
                if c.clip_grad: torch.nn.utils.clip_grad_norm_(fmodel.parameters(), c.clip_grad)
                opt.step(); opt.zero_grad(set_to_none=True); step += 1
        tl = (loss_sum / max(1, n_micro)).item()
        if not math.isfinite(tl): raise FloatingPointError("train loss is NaN/inf (try lower lr)")
        return tl, step, seen

    @torch.inference_mode()
    def _evaluate(self, fmodel, loader):
        fmodel.eval(); c1 = c5 = n = 0; ls = 0.0
        for x, y in loader:
            x = x.to(self.device, non_blocking=True); y = y.to(self.device, non_blocking=True)
            x = normalize(x.permute(0, 3, 1, 2).float()).contiguous(memory_format=torch.channels_last)
            with torch.autocast(self.device, dtype=self.amp_dtype):
                out = fmodel(x)
            out = out.float(); ls += F.cross_entropy(out, y, reduction="sum").item()
            top5 = out.topk(5, 1).indices; hit = top5 == y[:, None]
            c1 += hit[:, 0].sum().item(); c5 += hit.any(1).sum().item(); n += len(y)
        return 100 * c1 / n, 100 * c5 / n, ls / n

    # ---- main entry ----
    def run(self) -> dict:
        spec, c, st = self.spec, self.cfg, self.store
        self._seed(c.seed)
        model, source = build_model(spec, c.num_classes)
        params_m, gmacs = complexity(model, 224)
        model = model.to(self.device).to(memory_format=torch.channels_last)
        fmodel = torch.compile(model) if c.compile else model
        opt = torch.optim.AdamW(self._param_groups(model), lr=c.lr, fused=(self.device == "cuda"))

        train_dl = CachedLoader("train", c.train_res, self.micro_bs, True, c.seed, c.limit_batches)
        val_dl = CachedLoader("val", c.eval_res, 500, False, 0, c.limit_batches)
        steps_per_epoch = max(1, len(train_dl) // self.accum)

        start_epoch, step, best = 0, 0, (-1.0, 0)
        rp = st.resume_ckpt(spec.name)
        if rp.exists():
            ck = torch.load(rp, map_location=self.device, weights_only=False)
            if ck.get("recipe_hash") == c.recipe_hash():
                model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
                start_epoch, step, best = ck["epoch"], ck["step"], tuple(ck["best"])
                print(f"  resumed {spec.name} from epoch {start_epoch}")
            else:
                print("  resume file has a different recipe hash -> starting fresh")
        st.drop_epochs(spec.name, after_epoch=start_epoch)

        if self.device == "cuda": torch.cuda.reset_peak_memory_stats()
        t_train = 0.0; top1 = top5 = vloss = float("nan")
        for epoch in range(start_epoch, c.epochs):
            if self.device == "cuda": torch.cuda.synchronize()
            t0 = time.time()
            tl, step, seen = self._train_epoch(fmodel, opt, train_dl, epoch, step, steps_per_epoch)
            if self.device == "cuda": torch.cuda.synchronize()
            dt = time.time() - t0; t_train += dt
            top1, top5, vloss = self._evaluate(fmodel, val_dl)
            if top1 > best[0]: best = (top1, epoch + 1)
            st.append_epoch(dict(model=spec.name, epoch=epoch + 1, train_loss=tl, val_loss=vloss, top1=top1, top5=top5,
                                 lr=opt.param_groups[0]["lr"], epoch_time_s=dt, img_per_s=seen / dt))
            print(f"  [{spec.name}] ep {epoch+1:3d}/{c.epochs} loss {tl:.3f} | top1 {top1:.2f} top5 {top5:.2f} | {dt:.0f}s ({seen/dt:.0f} img/s)")
            torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), epoch=epoch + 1, step=step,
                            best=best, recipe_hash=c.recipe_hash()), rp)

        final = dict(model=model.state_dict(), name=spec.name, source=source, kwargs=spec.kwargs,
                     num_classes=c.num_classes, classes=json.loads((CACHE_DIR / "classes.json").read_text()),
                     config=asdict(c), recipe_hash=c.recipe_hash(), top1=top1, top5=top5)
        torch.save(final, st.final_ckpt(spec.name))
        if rp.exists(): rp.unlink()
        n_ep = max(1, c.epochs - start_epoch)
        return dict(model=spec.name, source=source, status="completed", recipe_hash=c.recipe_hash(),
                    epochs=c.epochs, top1=top1, top5=top5, val_loss=vloss, best_top1=best[0], best_epoch=best[1],
                    params_M=round(params_m, 3), GMACs_224=round(gmacs, 4),
                    train_res=c.train_res, eval_res=c.eval_res, batch_size=c.batch_size, micro_batch_size=self.micro_bs,
                    train_time_min=round(t_train / 60, 2), avg_epoch_s=round(t_train / n_ep, 1),
                    peak_vram_gb=round(torch.cuda.max_memory_allocated() / 2**30, 2) if self.device == "cuda" else None,
                    seed=c.seed, amp_dtype=c.amp_dtype, kwargs=json.dumps(spec.kwargs),
                    gpu=torch.cuda.get_device_name(0) if self.device == "cuda" else "cpu",
                    torch=torch.__version__, timm=timm.__version__, torchvision=torchvision.__version__,
                    checkpoint=str(st.final_ckpt(spec.name).relative_to(ROOT)),
                    finished_at=time.strftime("%Y-%m-%d %H:%M:%S"), error="")

## 6. Driver: trains only what is missing

In [17]:
def run_all(models=MODELS, cfg: TrainConfig = CFG, force=()):
    """force: model names to retrain even if already completed."""
    ensure_caches(cfg)
    store = ResultsStore(cfg)
    specs = [as_spec(m) for m in models]
    pending = [s for s in specs if s.name in force or not store.is_done(s.name, cfg.recipe_hash())]
    skipped = [s.name for s in specs if s not in pending]
    print(f"recipe {cfg.recipe_hash()} | skipping {len(skipped)} done: {skipped}")
    print(f"to train ({len(pending)}): {[s.name for s in pending]}")
    for s in pending:
        print(f"\n=== {s.name} ===")
        try:
            row = ModelTrainer(s, cfg, store).run()
        except Exception as e:
            oom = isinstance(e, torch.cuda.OutOfMemoryError)
            msg = ("CUDA OOM -> set micro_batch_size for this model" if oom else f"{type(e).__name__}: {e}")
            print(f"  FAILED: {msg}"); traceback.print_exc()
            row = dict(model=s.name, status="failed", recipe_hash=cfg.recipe_hash(), error=msg[:300],
                       finished_at=time.strftime("%Y-%m-%d %H:%M:%S"))
        store.upsert_summary(row)
        gc.collect(); torch.cuda.empty_cache()
    return store.summary()

## 7. Run

In [18]:
# 7a. Check that every model resolves, and see params / GMACs (no training)
preview_models()

W1021 19:52:28.448000 21344 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


,model,source,params_M,GMACs_224
0,mobilenetv1_100,timm,3.31,0.568
1,mobilenetv2_100,timm,2.35,0.300
2,mobilenetv3_small_100,timm,1.62,0.056
3,mobilenetv3_large_100,timm,4.33,0.215
4,mobilenetv4_conv_small,timm,2.62,0.185
5,mobilenetv4_hybrid_medium,timm,9.92,0.965
6,efficientnet_lite0,timm,3.50,0.384
7,efficientnet_b0,timm,4.14,0.385
8,ghostnet_100,timm,4.03,0.140
9,ghostnetv2_100,timm,5.00,0.166


In [ ]:
# 7b. Smoke test (2 epochs, 20 batches, writes to _debug folders). Run this first after any code change.
DEBUG_CFG = replace(CFG, epochs=2, warmup_epochs=1, limit_batches=20, debug=True)
run_all(MODELS, DEBUG_CFG, force=[as_spec(m).name for m in MODELS])

train: 130000 images, 100 classes


cache train@160:   0%|                                                                         | 0/508 [00:00<?, ?it/s]

In [ ]:
# 7c. Real run. Safe to re-run: completed models are skipped, interrupted ones resume.
results = run_all(MODELS, CFG)
results

In [ ]:
# 7d. Inspect results
store = ResultsStore(CFG)
store.summary().sort_values("top1", ascending=False)

In [ ]:
# Example: load a trained checkpoint
# ck = torch.load(CKPT_DIR / "mobilenetv2_100.pt", map_location="cpu", weights_only=False)
# net, _ = build_model(ModelSpec(ck["name"], ck["source"], ck["kwargs"]), ck["num_classes"]); net.load_state_dict(ck["model"])